# RLHF with PPO — Simple Colab Lab Demonstration

This notebook demonstrates the basic idea of **Reinforcement Learning from Human Feedback (RLHF)** using **PPO (Proximal Policy Optimization)**.

### Lab flow
1. Load a small language model.
2. Give prompts to the model.
3. Use a simple **reward function** as a stand-in for human feedback.
4. Optimize the model with PPO.
5. Compare responses before and after training.

> This is an educational demonstration. The reward is rule-based rather than collected from real human annotators.


In [ ]:
# Install a compatible, simple PPO stack (peft added for LoRA)
!pip -q install "trl==0.8.6" "transformers<4.46" "accelerate<1.0" datasets peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 13.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip -q install peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.9 MB/s eta 0:00:00


## 1. Load a small language model

We use `HuggingFaceTB/SmolLM2-1.7B-Instruct` (~1.7B parameters, about 3x larger than Qwen2.5-0.5B, so its answers are far more coherent).
To keep PPO affordable on a free Colab T4 GPU, we train only small **LoRA adapters** instead of all weights.

> Use a GPU runtime: Runtime → Change runtime type → T4 GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization to prevent VRAM crashes with PPO
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.chat_template = "{% for message in messages %}{{ message['content'] }}{% endfor %}"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)
base_model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded:", MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


## 2. Create a tiny prompt dataset

For a real RLHF system, prompts would come from a larger dataset. Here we use only a few prompts.


In [ ]:
prompts = [
    "Explain artificial intelligence in simple words.",
    "What is machine learning?",
    "Explain a neural network.",
    "What is reinforcement learning?",
    "Give one benefit of artificial intelligence.",
    "What is a reward in reinforcement learning?",
    "How does a model learn from data?",
    "What is an algorithm?",
    "Explain what an agent is in AI.",
    "Why is data important for machine learning?",
    "What is deep learning?",
    "Explain supervised learning.",
    "What is training data?",
    "How do neural networks make predictions?",
    "What is an environment in reinforcement learning?",
    "Explain AI to a ten year old.",
]

# Instruct models expect the chat format, so wrap each prompt with the chat template
def build_prompt(p):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": p + " Answer in 2 short sentences."}],
        tokenize=False,
        add_generation_prompt=True
    )

print("Number of prompts:", len(prompts))

Number of prompts: 16


## 3. Define a simple reward function

Real RLHF normally uses human preference data and a learned reward model.

For this lab, we use a simple automatic reward function. A response gets a higher reward when it contains useful AI/ML terms and has a reasonable length.


In [ ]:
GOOD_WORDS = {
    "ai", "artificial", "intelligence", "machine", "learning",
    "model", "data", "algorithm", "neural", "network",
    "reinforcement", "reward", "agent", "environment"
}

def reward_function(text):
    words = [w.strip(".,!?;:()\"'").lower() for w in text.split()]
    if not words:
        return -1.0

    # distinct useful terms (capped, so repeating "AI AI AI" does not pay off)
    keyword_score = min(len({w for w in words if w in GOOD_WORDS}), 6) * 0.5

    # reasonable length
    length_bonus = min(len(words) / 30.0, 1.0)

    # finishing the sentence cleanly
    clean_end = 0.5 if text.strip().endswith((".", "!", "?")) else 0.0

    # penalise repetition
    repetition_penalty = (1 - len(set(words)) / len(words)) * 2.0

    return float(keyword_score + length_bonus + clean_end - repetition_penalty)

example = "Machine learning uses data to train a model."
print("Example:", example)
print("Reward:", reward_function(example))

Example: Machine learning uses data to train a model.
Reward: 2.7666666666666666


## 4. See the model before PPO

This gives us a baseline response before reinforcement learning.


In [ ]:
def generate_response(model, prompt, max_new_tokens=48):
    inputs = tokenizer(build_prompt(prompt), return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            temperature=0.7,
            pad_token_id=tokenizer.pad_token_id
        )

    # decode only the newly generated tokens (not the prompt)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


baseline = {}
for prompt in prompts[:5]:
    response = generate_response(base_model, prompt)
    baseline[prompt] = response

    print("PROMPT:", prompt)
    print("RESPONSE:", response)
    print("REWARD:", reward_function(response))
    print("-" * 70)

# free GPU memory before PPO
import gc
del base_model
gc.collect()
torch.cuda.empty_cache()

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


PROMPT: Explain artificial intelligence in simple words.
RESPONSE:  Artificial Intelligence (AI) is a computer system that can perform tasks requiring human intelligence, like learning and decision-making, without explicit programming. It uses algorithms to process data and make decisions or predictions based on patterns learned from the data. AI systems
REWARD: 3.3048780487804876
----------------------------------------------------------------------
PROMPT: What is machine learning?
RESPONSE:  Machine learning is a subset of artificial intelligence that enables computer systems to learn and improve from experience or data without being explicitly programmed.
Machine learning involves the use of algorithms and statistical models to enable computers to learn from data, recognize patterns, and make
REWARD: 3.0555555555555554
----------------------------------------------------------------------
PROMPT: Explain a neural network.
RESPONSE:  A neural network is an artificial intelligence mo

## 5. Create the PPO model

PPO needs:
- a policy model,
- a reference model,
- and a value head.

The reference model provides a stable reference while the policy is updated.


In [ ]:
from datasets import Dataset
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
import torch

# 1. Configure 4-bit quantization to fit within T4 VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# 2. Load model with value head and quantization enabled
policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

# 3. Apply PEFT LoRA configuration
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
policy_model.pretrained_model = get_peft_model(policy_model.pretrained_model, peft_config)

# 4. Prompt dataset mapping with reduced max length
dataset = Dataset.from_dict({"query": prompts})

def tokenize(example):
    encoded = tokenizer(
        example["query"],
        truncation=True,
        max_length=32
    )
    return {
        "input_ids": encoded["input_ids"],
        "query": example["query"]
    }

dataset = dataset.map(tokenize)

def collator(data):
    return {
        "input_ids": [
            torch.tensor(x["input_ids"], dtype=torch.long)
            for x in data
        ],
        "query": [x["query"] for x in data]
    }

ppo_config = PPOConfig(
    model_name=MODEL_NAME,
    learning_rate=1e-5,
    batch_size=1,
    mini_batch_size=1,
    ppo_epochs=1,
    seed=42
)

# 5. Initialize PPOTrainer
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    tokenizer=tokenizer,
    dataset=dataset,
    data_collator=collator
)

print("PPO trainer ready with 4-bit quantization and PEFT.")

## 6. Run a few PPO updates

The simplified training loop is:

`prompt → policy response → reward → PPO update → updated policy`

Only a few updates are used because this is a lab demonstration.


In [ ]:
generation_kwargs = {
    "max_new_tokens": 48,
    "do_sample": True,
    "top_k": 50,
    "temperature": 0.7,
    "pad_token_id": tokenizer.eos_token_id
}

NUM_EPOCHS = 8   # each epoch = 2 batches of 8 prompts -> 16 PPO updates

for epoch in range(NUM_EPOCHS):
    for step, batch in enumerate(ppo_trainer.dataloader):

        query_tensors = batch["input_ids"]

        response_tensors = ppo_trainer.generate(
            query_tensors,
            return_prompt=False,
            batch_size=4,
            **generation_kwargs
        )

        responses = [
            tokenizer.decode(r, skip_special_tokens=True)
            for r in response_tensors
        ]

        rewards = [
            torch.tensor(
                reward_function(response),
                dtype=torch.float32
            )
            for response in responses
        ]

        stats = ppo_trainer.step(
            query_tensors,
            response_tensors,
            rewards
        )

        mean_reward = sum(r.item() for r in rewards) / len(rewards)
        print(f"Epoch {epoch + 1} | Step {step + 1} | Mean reward: {mean_reward:.2f}")
        print("Prompt:", tokenizer.decode(query_tensors[0], skip_special_tokens=True).split("user")[-1].strip()[:80])
        print("Response:", responses[0])
        print("-" * 70)

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.13/dist-packages/trl/trainer/ppo_trainer.py:1285: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  std_scores = data["scores"].std()
/usr/local/lib/python3.13/dist-packages/trl/trainer/ppo_trainer.py:1312: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  stats["tokens/queries_len_std"] = torch.std(query_lens).cpu().numpy().item()
/usr/local/lib/python3.13/dist-packages/trl/trainer/ppo_trainer.

Epoch 1 | Step 1 | Mean reward: 1.78
Prompt: Explain artificial intelligence in simple words.
Response:  AI is like teaching a computer to learn and understand things on its own, just like how you might teach yourself something new. It uses algorithms and data to make decisions or perform tasks without being explicitly programmed to do so. So instead of following
----------------------------------------------------------------------
Epoch 1 | Step 2 | Mean reward: 3.22
Prompt: What is a reward in reinforcement learning?
Response:  In reinforcement learning, a reward is an intrinsic or extrinsic feedback signal that indicates the agent's progress towards its goals. It can be used to adjust the action of the agent based on the outcome of each action taken during the training process.
----------------------------------------------------------------------
Epoch 1 | Step 3 | Mean reward: 1.13
Prompt: What is an algorithm?
Response:  An algorithm is a set of instructions that can be followed

## 7. Test the trained policy


In [ ]:
def generate_trained(prompt):
    # Tokenize prompt
    inputs = tokenizer(
        build_prompt(prompt),
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # PPOTrainer expects a 1-D query tensor
    device = next(ppo_trainer.model.parameters()).device
    query = inputs["input_ids"][0].to(device)

    # Generate with PPO policy
    response = ppo_trainer.generate(
        query,
        return_prompt=False,
        max_new_tokens=48,
        do_sample=True,
        top_k=50,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    # -----------------------------------------
    # Robustly convert PPO output to 1-D tensor
    # -----------------------------------------

    if isinstance(response, list):
        response = response[0]

    if not torch.is_tensor(response):
        response = torch.tensor(response)

    # Remove batch dimension if present
    if response.ndim > 1:
        response = response[0]

    # Move to CPU and convert to Python list
    response_ids = response.detach().cpu().tolist()

    # Make sure we have a flat list of token IDs
    if response_ids and isinstance(response_ids[0], list):
        response_ids = response_ids[0]

    return tokenizer.decode(
        response_ids,
        skip_special_tokens=True
    )


# =========================================
# Test trained policy
# =========================================

for prompt in prompts:

    response = generate_trained(prompt)

    print("PROMPT:", prompt)
    if prompt in baseline:
        print("BEFORE PPO:", baseline[prompt], "| reward:", round(reward_function(baseline[prompt]), 2))
    print("TRAINED RESPONSE:", response)
    print("REWARD:", reward_function(response))
    print("-" * 70)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


PROMPT: Explain artificial intelligence in simple words.
BEFORE PPO:  Artificial Intelligence (AI) is a computer system that can perform tasks requiring human intelligence, like learning and decision-making, without explicit programming. It uses algorithms to process data and make decisions or predictions based on patterns learned from the data. AI systems | reward: 3.3
TRAINED RESPONSE:  Artificial Intelligence (AI) is like teaching a computer to do tasks that usually require human intelligence, such as understanding language or making decisions. AI can be used for things like helping doctors diagnose illnesses, improving customer service, and even designing new products
REWARD: 2.357142857142857
----------------------------------------------------------------------
PROMPT: What is machine learning?
BEFORE PPO:  Machine learning is a subset of artificial intelligence that enables computer systems to learn and improve from experience or data without being explicitly programmed.
Machine

## 8. RLHF pipeline

```text
Prompt
   ↓
Language Model / Policy
   ↓
Generated Response
   ↓
Reward Function
   ↓
Reward Signal
   ↓
PPO
   ↓
Updated Policy
```

### Key concepts

- **Policy:** the language model being optimized.
- **Reference model:** a copy used as a stable reference.
- **Reward:** a scalar describing how desirable the response is.
- **PPO:** updates the policy while constraining the update.
- **Real RLHF:** typically uses human preference data and a learned reward model rather than the simple keyword reward used here.
